# Batch Effect Correction Methods

This notebook applies three batch correction methods to Cell Painting data:
1. **FastMNN** — fast mutual nearest neighbors (batchelor)
2. **Seurat CCA** — canonical correlation analysis (Seurat)
3. **Seurat RPCA** — reciprocal PCA (Seurat)

## Setup & Configuration

In [3]:
# ---- Libraries ----
library(arrow)
library(batchelor)
library(SingleCellExperiment)
library(Seurat)
library(dplyr)
library(Matrix)

# ---- Paths ----
input_file <- '/absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata/MAD_adata.parquet'
output_dir <- '/absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata/'

# ---- Parameters ----
batch_col <- 'Metadata_Source'  # column used for batch labels
# batch_col <- 'Metadata_Label'  # alternative

cat(sprintf('Input:  %s\n', input_file))
cat(sprintf('Output: %s\n', output_dir))
cat(sprintf('Batch column: %s\n', batch_col))

Input:  /absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata/MAD_adata.parquet
Output: /absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata/
Batch column: Metadata_Source


## Data Loading

Read parquet file, split into metadata columns and feature columns.

In [4]:
# ---- Read & split ----
parquet_data <- read_parquet(input_file)
col_names     <- names(parquet_data)
metadata_cols <- col_names[grepl("^Metadata_", col_names)]
features_cols <- col_names[!grepl("^Metadata_", col_names)]

features <- parquet_data[, features_cols, drop = FALSE]
metadata <- parquet_data[, metadata_cols, drop = FALSE]
batch_info <- parquet_data[, batch_col, drop = TRUE]

cat(sprintf("Metadata columns: %d\n", length(metadata_cols)))
cat(sprintf("Feature columns:  %d\n", length(features_cols)))
cat(sprintf("Total rows:       %d\n", nrow(parquet_data)))
cat(sprintf("Unique batches:   %d\n", length(unique(batch_info))))

Metadata columns: 5
Feature columns:  3556
Total rows:       50256
Unique batches:   10


## Helper: build Seurat objects per batch

Splits data by batch, creates a Seurat object for each, and runs PCA.

In [5]:
build_seurat_list <- function(features, metadata, batch_info) {
  batches     <- split(features, batch_info)
  meta_batches <- split(metadata, batch_info)
  
  seurat_list <- lapply(seq_along(batches), function(i) {
    raw  <- as.data.frame(batches[[i]])
    meta <- as.data.frame(meta_batches[[i]])
    
    # Seurat requires row names without underscores in feature names
    names(raw) <- gsub("_", "-", names(raw))
    
    obj <- CreateSeuratObject(counts = t(raw), meta.data = meta)
    obj <- SetAssayData(obj, layer = "data",       new.data = t(raw))
    obj <- SetAssayData(obj, layer = "scale.data",  new.data = t(raw))
    obj <- RunPCA(obj, features = colnames(raw), verbose = FALSE)
    obj
  })
  
  # Remove features with zero / non-finite variance from each object
  for (i in seq_along(seurat_list)) {
    counts <- GetAssayData(seurat_list[[i]], layer = "data")
    keep   <- apply(counts, 1, function(x) var(x, na.rm = TRUE) > 0 && all(is.finite(x)))
    seurat_list[[i]] <- seurat_list[[i]][keep, ]
  }
  
  seurat_list
}

## Helper: extract & write corrected data

Extracts the integrated assay, transposes back to samples × features, and writes parquet.

In [6]:
write_seurat_result <- function(integrated_obj, output_file) {
  int_data <- GetAssayData(integrated_obj, assay = "integrated", layer = "data")
  corrected <- int_data %>%
    as.matrix() %>%
    t() %>%
    as.data.frame()
  colnames(corrected) <- gsub("-", "_", colnames(corrected))
  
  # Drop Seurat-internal columns (first 3)
  meta_corrected <- integrated_obj@meta.data[, -c(1:3)]
  corrected_df   <- cbind(meta_corrected, corrected)
  
  write_parquet(corrected_df, output_file)
  cat(sprintf("Saved: %s  (%d rows × %d cols)\n",
              output_file, nrow(corrected_df), ncol(corrected_df)))
}

---
## Method 1: FastMNN (batchelor)

In [7]:
cat("Running FastMNN ...\n")

corrected <- fastMNN(t(features), batch = batch_info)
corrected_df <- cbind(metadata, reducedDim(corrected))

output_file <- file.path(output_dir, "Fast_MNN.parquet")
write_parquet(corrected_df, output_file)

cat(sprintf("Saved: %s  (%d rows × %d cols)\n",
            output_file, nrow(corrected_df), ncol(corrected_df)))
head(corrected_df[, 1:min(8, ncol(corrected_df))])

Running FastMNN ...
Saved: /absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata//Fast_MNN.parquet  (50256 rows × 55 cols)


,Metadata_Plate,Metadata_JCP2022,Metadata_Well,Metadata_Batch,Metadata_Source,1,2,3
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,Dest210726-160150,JCP2022_043547,A01,2021_08_03_U2OS_48_hr_run12,source_10,-0.007002401,0.2677784,0.40254504
2,Dest210726-160150,JCP2022_050797,A02,2021_08_03_U2OS_48_hr_run12,source_10,0.127591507,0.4527839,0.43896639
3,Dest210726-160150,JCP2022_050997,A03,2021_08_03_U2OS_48_hr_run12,source_10,0.034272106,0.5167408,0.03042562
4,Dest210726-160150,JCP2022_108326,A04,2021_08_03_U2OS_48_hr_run12,source_10,-0.329843518,0.3974519,0.27492019
5,Dest210726-160150,JCP2022_033924,A05,2021_08_03_U2OS_48_hr_run12,source_10,-0.479368682,0.2807539,0.17981758
6,Dest210726-160150,JCP2022_027911,A06,2021_08_03_U2OS_48_hr_run12,source_10,-0.267381032,0.3181370,0.31226291


---
## Method 2: Seurat CCA

In [8]:
cat("Running Seurat CCA ...\n")

seurat_method <- "cca"
seurat_list   <- build_seurat_list(features, metadata, batch_info)

cat(sprintf("Built %d Seurat objects\n", length(seurat_list)))

# Find anchors & integrate
options(future.globals.maxSize = Inf)
anchor_set <- FindIntegrationAnchors(
  object.list      = seurat_list,
  reduction        = seurat_method,
  anchor.features  = nrow(GetAssayData(seurat_list[[1]])),
  scale            = FALSE,
  verbose          = TRUE
)
integrated_obj <- IntegrateData(anchor_set, verbose = TRUE)

# Write
output_file <- file.path(output_dir, "Seurat_cca_adata.parquet")
write_seurat_result(integrated_obj, output_file)

Running Seurat CCA ...


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


Built 10 Seurat objects


Warning message in CheckDuplicateCellNames(object.list = object.list):
“Some cell names are duplicated across objects provided. Renaming to enforce unique cell names.”
Computing 3556 integration features

No variable features found for object1 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

Warning message in eval(predvars, data, env):
“产生了NaNs”
Warning message in hvf.info$variance.expected[not.const] <- 10^fit$fitted:
“被替换的项目不是替换值长度的倍数”
No variable features found for object2 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

Warning message in eval(predvars, data, env):
“产生了NaNs”
Warning message in hvf.info$variance.expected[not.const] <- 10^fit$fitted:
“被替换的项目不是替换值长度的倍数”
No variable features found for object3 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

Warning message in eval(predvars, data, env):
“产生了NaNs”
Warning message in hvf.info$

Saved: /absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata//Seurat_cca_adata.parquet  (50256 rows × 3535 cols)


---
## Method 3: Seurat RPCA

In [9]:
cat("Running Seurat RPCA ...\n")

seurat_method <- "rpca"
seurat_list   <- build_seurat_list(features, metadata, batch_info)

cat(sprintf("Built %d Seurat objects\n", length(seurat_list)))

# Find anchors & integrate
options(future.globals.maxSize = Inf)
anchor_set <- FindIntegrationAnchors(
  object.list      = seurat_list,
  reduction        = seurat_method,
  anchor.features  = nrow(GetAssayData(seurat_list[[1]])),
  scale            = FALSE,
  verbose          = TRUE
)
integrated_obj <- IntegrateData(anchor_set, verbose = TRUE)

# Write
output_file <- file.path(output_dir, "Seurat_rpca_adata.parquet")
write_seurat_result(integrated_obj, output_file)

Running Seurat RPCA ...


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


Built 10 Seurat objects


Warning message in CheckDuplicateCellNames(object.list = object.list):
“Some cell names are duplicated across objects provided. Renaming to enforce unique cell names.”
Computing 3556 integration features

No variable features found for object1 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

Warning message in eval(predvars, data, env):
“产生了NaNs”
Warning message in hvf.info$variance.expected[not.const] <- 10^fit$fitted:
“被替换的项目不是替换值长度的倍数”
No variable features found for object2 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

Warning message in eval(predvars, data, env):
“产生了NaNs”
Warning message in hvf.info$variance.expected[not.const] <- 10^fit$fitted:
“被替换的项目不是替换值长度的倍数”
No variable features found for object3 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

Warning message in eval(predvars, data, env):
“产生了NaNs”
Warning message in hvf.info$

Saved: /absolute/path/outputs/05_batch_effect/Target_2_Compoud/batch_adata//Seurat_rpca_adata.parquet  (50256 rows × 3535 cols)


---
## Session Info

In [ ]:
sessionInfo()